# Tutorial 03 — Model Components

Tests each model building block with synthetic tensors:

1. **Poolers** — 6 CDR pooling strategies → `(B, K, d_e)`
2. **Projectors** — MLP, Q-Former, Perceiver → `(B, K_out, d_out)`
3. **AntibodyDecoder** — GPT forward pass + generation
4. **AbLLaVA** — end-to-end with synthetic embeddings
5. **Pooling comparison** — visualise K and parameter counts

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import torch
import torch.nn as nn
torch.manual_seed(42)

## 1. Pooling strategies

All poolers share the same signature:
```
forward(struct_emb, cdr_spans, pad_mask, plddt=None) -> (B, K, d_e)
```

In [ ]:
from src.models.pooling import POOLERS, build_pooler

B, N, d_e = 2, 230, 512
struct_emb = torch.randn(B, N, d_e)
pad_mask   = torch.ones(B, N, dtype=torch.bool)
plddt      = torch.rand(B, N) * 30 + 70   # uniform [70, 100]

# Synthetic CDR spans: 6 CDRs per sample
cdr_spans = torch.tensor([
    [[26,32],[50,57],[95,102],[143,153],[169,173],[208,216]],
    [[27,33],[51,58],[96,103],[144,154],[170,174],[209,217]],
], dtype=torch.long)

print(f"Input:  struct_emb {struct_emb.shape}, cdr_spans {cdr_spans.shape}")
print()

results = {}
for name, cls in POOLERS.items():
    if name == 'cdr_attention':
        pooler = cls(d_e)
    elif name == 'cdr_segmented':
        pooler = cls(n_segments=3)
    else:
        pooler = cls()
    pooler.eval()
    with torch.no_grad():
        out = pooler(struct_emb, cdr_spans, pad_mask, plddt)
    results[name] = out
    n_params = sum(p.numel() for p in pooler.parameters())
    print(f"  {name:20s}  output {str(out.shape):20s}  params={n_params:,}")

### Visualise pooling output sizes

In [ ]:
try:
    import matplotlib.pyplot as plt

    names = list(results.keys())
    K_vals = [results[n].shape[1] for n in names]

    fig, ax = plt.subplots(figsize=(8, 3))
    bars = ax.barh(names, K_vals, color='steelblue')
    ax.bar_label(bars, fmt='%d', padding=3)
    ax.set_xlabel("K  (number of prefix tokens passed to projector)")
    ax.set_title("Pooler output token counts")
    plt.tight_layout()
    plt.savefig("pooler_K.png", dpi=120)
    plt.show()
    print("Saved pooler_K.png")
except ImportError:
    print("matplotlib not installed — skipping plot")
    for n, out in results.items():
        print(f"  {n}: K={out.shape[1]}")

## 2. Projectors — compress pooled tokens to decoder dimension

In [ ]:
from src.models.projectors import PROJECTORS, build_projector

d_in  = d_e    # 512  (AntiFold embedding dim)
d_out = 1024   # decoder d_model
K_in  = 6      # from cdr_mean pooler

pooled = torch.randn(B, K_in, d_in)

configs = {
    'mlp':       dict(d_in=d_in, d_out=d_out),
    'qformer':   dict(d_in=d_in, d_out=d_out, n_queries=32, n_layers=2, n_heads=8),
    'perceiver': dict(d_in=d_in, d_out=d_out, n_latents=32, n_layers=1, n_heads=8),
}

print(f"Input pooled: {pooled.shape}\n")
for name, cfg in configs.items():
    proj = PROJECTORS[name](**cfg)
    proj.eval()
    with torch.no_grad():
        out = proj(pooled)
    n_params = sum(p.numel() for p in proj.parameters())
    print(f"  {name:10s}  {str(pooled.shape)} -> {str(out.shape)}  params={n_params/1e6:.2f}M")

## 3. AntibodyDecoder — GPT-style forward + generation

In [ ]:
from src.models.decoder import AntibodyDecoder
from src.utils.tokenizer import AntibodyTokenizer

tok = AntibodyTokenizer()

# Tiny decoder for quick testing
decoder = AntibodyDecoder(
    vocab_size=tok.vocab_size,
    d_model=128,
    n_layers=2,
    n_heads=4,
    d_ff=256,
    max_seq_len=256,
)
decoder.eval()

n_params = sum(p.numel() for p in decoder.parameters())
print(f"Decoder params: {n_params/1e6:.2f}M  (tiny test config)")

# --- Forward pass with input_ids ---
B_d, L = 2, 30
input_ids = torch.randint(5, tok.vocab_size, (B_d, L))
labels    = input_ids.clone()

out = decoder(input_ids=input_ids, labels=labels)
print(f"\nForward output keys: {list(out.keys())}")
print(f"  logits: {out['logits'].shape}")
print(f"  loss  : {out['loss'].item():.4f}")

In [ ]:
# --- Forward with inputs_embeds (used by AbLLaVA prefix injection) ---
prefix_len = 8
prefix_emb = torch.randn(B_d, prefix_len, 128)
token_emb  = decoder.embed_tokens(input_ids)
combined   = torch.cat([prefix_emb, token_emb], dim=1)  # (B, prefix+L, d_model)

out2 = decoder(inputs_embeds=combined)
print(f"inputs_embeds forward: logits {out2['logits'].shape}  (L+prefix={combined.shape[1]})")

In [ ]:
# --- Generation ---
prefix = torch.randn(1, prefix_len, 128)
gen_ids = decoder.generate(
    prefix_embeds=prefix,
    bos_id=tok.bos_id,
    eos_id=tok.eos_id,
    max_new_tokens=40,
    temperature=1.0,
    top_p=0.9,
    do_sample=True,
)
print(f"Generated token ids: {gen_ids.tolist()[0][:20]}...")
heavy, light = tok.split_pair_ids(gen_ids.tolist()[0])
print(f"Heavy: {heavy or '(empty)'}")
print(f"Light: {light or '(empty)'}")

## 4. AbLLaVA — end-to-end pipeline

In [ ]:
from src.models.pooling import build_pooler
from src.models.projectors import build_projector
from src.models.abllava import AbLLaVA

d_model = 128

pooler    = build_pooler('cdr_mean')
projector = build_projector('mlp', d_in=512, d_out=d_model)
decoder   = AntibodyDecoder(
    vocab_size=tok.vocab_size,
    d_model=d_model,
    n_layers=2, n_heads=4, d_ff=256,
)

model = AbLLaVA(decoder=decoder, projector=projector, pooler_name='cdr_mean')
model.eval()

# Build a synthetic batch
batch = {
    'struct_emb': torch.randn(B, N, 512),
    'cdr_spans': cdr_spans,
    'pad_mask': torch.ones(B, N, dtype=torch.bool),
    'plddt': torch.rand(B, N) * 30 + 70,
    'seq_ids': torch.randint(5, tok.vocab_size, (B, 50)),
    'seq_pad_mask': torch.ones(B, 50, dtype=torch.long),
}

with torch.no_grad():
    out = model(batch)

print(f"AbLLaVA forward: logits {out['logits'].shape}  loss={out['loss'].item():.4f}")

## 5. LoRA application

In [ ]:
from src.models.lora import apply_lora

decoder_for_lora = AntibodyDecoder(
    vocab_size=tok.vocab_size, d_model=128, n_layers=2, n_heads=4, d_ff=256
)
before = sum(p.numel() for p in decoder_for_lora.parameters() if p.requires_grad)

lora_decoder = apply_lora(decoder_for_lora, r=4, alpha=16, dropout=0.05,
                           targets=["q_proj", "v_proj"])

trainable  = sum(p.numel() for p in lora_decoder.parameters() if p.requires_grad)
total      = sum(p.numel() for p in lora_decoder.parameters())
print(f"Trainable params after LoRA: {trainable:,} / {total:,}  ({100*trainable/total:.1f}%)")

---
**Summary**: Each module forwards cleanly on random tensors. The full pipeline is `struct_emb → pooler → projector → concat with token embeddings → decoder`.